# Video Game Sales & Engagement Analysis - PHASE 1

In [1]:
import pandas as pd
import numpy as np

IMPORT GAMES DATA TO DATAFRAME

In [2]:
df_games = pd.read_csv('games.csv')
df_games.head()

,Unnamed: 0,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist
0,0,Elden Ring,"Feb 25, 2022","['Bandai Namco Entertainment', 'FromSoftware']",4.5,3.9K,3.9K,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17K,3.8K,4.6K,4.8K
1,1,Hades,"Dec 10, 2019",['Supergiant Games'],4.3,2.9K,2.9K,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21K,3.2K,6.3K,3.6K
2,2,The Legend of Zelda: Breath of the Wild,"Mar 03, 2017","['Nintendo', 'Nintendo EPD Production Group No...",4.4,4.3K,4.3K,"['Adventure', 'RPG']",The Legend of Zelda: Breath of the Wild is the...,['This game is the game (that is not CS:GO) th...,30K,2.5K,5K,2.6K
3,3,Undertale,"Sep 15, 2015","['tobyfox', '8-4']",4.2,3.5K,3.5K,"['Adventure', 'Indie', 'RPG', 'Turn Based Stra...","A small child falls into the Underground, wher...",['soundtrack is tied for #1 with nier automata...,28K,679,4.9K,1.8K
4,4,Hollow Knight,"Feb 24, 2017",['Team Cherry'],4.4,3K,3K,"['Adventure', 'Indie', 'Platform']",A 2D metroidvania with an emphasis on close co...,"[""this games worldbuilding is incredible, with...",21K,2.4K,8.3K,2.3K


BASIC DATA EXPLORATION (GAME DATASET)

In [3]:
df_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1512 non-null   int64  
 1   Title              1512 non-null   object 
 2   Release Date       1512 non-null   object 
 3   Team               1511 non-null   object 
 4   Rating             1499 non-null   float64
 5   Times Listed       1512 non-null   object 
 6   Number of Reviews  1512 non-null   object 
 7   Genres             1512 non-null   object 
 8   Summary            1511 non-null   object 
 9   Reviews            1512 non-null   object 
 10  Plays              1512 non-null   object 
 11  Playing            1512 non-null   object 
 12  Backlogs           1512 non-null   object 
 13  Wishlist           1512 non-null   object 
dtypes: float64(1), int64(1), object(12)
memory usage: 165.5+ KB


In [4]:
df_games.isnull().sum()

Unnamed: 0            0
Title                 0
Release Date          0
Team                  1
Rating               13
Times Listed          0
Number of Reviews     0
Genres                0
Summary               1
Reviews               0
Plays                 0
Playing               0
Backlogs              0
Wishlist              0
dtype: int64

# Cleaning & Standardizing - NUMERICAL FIELDS

FIX 1: USELESS COLUMN

In [5]:
# index column accidentally saved from CSV
df_games.drop(columns=['Unnamed: 0'], inplace=True)

# drop() → removes columns or/ rows
# columns=[...] → specifies column removal
# inplace=True → modifies original DataFrame (memory efficient)

FIX 2: DATATYPE ISSUE

In [6]:
# Change release date from string object to datetime so we can filter by year and perform time intelligence analysis
df_games['Release Date'] = pd.to_datetime(
                        df_games['Release Date'],errors='coerce'
            )
# errors='coerce' → invalid dates become NaT (safe null) instead of crashing the code

df_games['Reviews'] = df_games['Reviews'].astype(str) # Convert Reviews to string for consistent processing


FIX 3: Convert K-values (thousand currency unit) to Numbers. 

In [7]:
# Created a reusable cleaning function to convert strings with 'K' suffix to numeric values. Handles mixed formats safely.

def convert_to_number(x): # x is the input value to convert
    if isinstance(x,str): # Check if x is a string using prebuilt function isinstance() with parameters (value to check, data type)
           x = x.strip() # Remove leading and trailing whitespace from a string. move ahead after string is cleaned 
           if x.endswith('K'): # Check if the string ends with 'K'
                return float(x.replace("K","")) * 1000 # Remove 'K'. replace with empty string and convert to float, then multiply by 1000
           else:
                return float(x) # Convert to float if it doesn't end with 'K'
    return x # Return the original value if it's not a string

In [8]:
# Appling it to columns in dataframe with mixed formats
engagement_cols = [
    'Times Listed',
    'Number of Reviews',
    'Plays',
    'Playing',
    'Backlogs',
    'Wishlist'
]

# Loop through each column in the engagement_cols list with col as the loop variable to reference the current column being processed
for col in engagement_cols: 
    df_games[col] = df_games[col].apply(convert_to_number)
    # Applied the conversion function to each specified column using apply() method

In [9]:
df_games[engagement_cols].head()

,Times Listed,Number of Reviews,Plays,Playing,Backlogs,Wishlist
0,3900.0,3900.0,17000.0,3800.0,4600.0,4800.0
1,2900.0,2900.0,21000.0,3200.0,6300.0,3600.0
2,4300.0,4300.0,30000.0,2500.0,5000.0,2600.0
3,3500.0,3500.0,28000.0,679.0,4900.0,1800.0
4,3000.0,3000.0,21000.0,2400.0,8300.0,2300.0


# Cleaning & Standardizing - CATEGORICAL FIELDS

FIX 4: Handle String Lists → Python list inside Column

In [10]:
import ast # Abstract Syntax Trees - safely evaluate string representations of Python objects (like lists, dicts) without using eval()

df_games['Team'] = df_games['Team'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x,str) else x
)

# ast.literal_eval() Safely converts string lists → Python lists
# "lambda" Applies logic row-by-row and isinstance(x,str) checks if the value is a string before conversion to avoid errors on non-string values


df_games['Genres'] = df_games['Genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x,str) else x
)

FIX 5: Handle missing value In Team Column

In [11]:
def clean_team(x):
    if isinstance(x, list):
        return x
    elif pd.isna(x):
        return ['Unknown']
    else:
        return x

df_games['Team'] = df_games['Team'].apply(clean_team)


In [12]:
df_games['Rating'] = df_games['Rating'].round(1) # Round ratings to 1 decimal place for cleaner presentation

In [13]:
df_games['Summary'].fillna('Not Available', inplace=True)

C:\Users\aniru\AppData\Local\Temp\ipykernel_8556\3069259477.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_games['Summary'].fillna('Not Available', inplace=True)


# SANITY CHECK / VALIDATING THE FIXATIONS

In [14]:
df_games.head(2)

,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist
0,Elden Ring,2022-02-25,"[Bandai Namco Entertainment, FromSoftware]",4.5,3900.0,3900.0,"[Adventure, RPG]","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0
1,Hades,2019-12-10,[Supergiant Games],4.3,2900.0,2900.0,"[Adventure, Brawler, Indie, RPG]",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21000.0,3200.0,6300.0,3600.0


In [15]:
df_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Title              1512 non-null   object        
 1   Release Date       1509 non-null   datetime64[ns]
 2   Team               1512 non-null   object        
 3   Rating             1499 non-null   float64       
 4   Times Listed       1512 non-null   float64       
 5   Number of Reviews  1512 non-null   float64       
 6   Genres             1512 non-null   object        
 7   Summary            1512 non-null   object        
 8   Reviews            1512 non-null   object        
 9   Plays              1512 non-null   float64       
 10  Playing            1512 non-null   float64       
 11  Backlogs           1512 non-null   float64       
 12  Wishlist           1512 non-null   float64       
dtypes: datetime64[ns](1), float64(7), object(5)
memory usage: 153.7

In [16]:
df_games[df_games.isnull().any(axis=1)]


,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist
587,Final Fantasy XVI,2023-06-22,"[Square Enix, Square Enix Creative Business Un...",NaN,422.0,422.0,[RPG],Final Fantasy XVI is an upcoming action role-p...,[],37.0,10.0,732.0,2400.0
644,Deltarune,NaT,[tobyfox],4.3,313.0,313.0,"[Adventure, Indie, Music, Puzzle, RPG]","UNDERTALE's parallel story, DELTARUNE. Meet ne...","['Spamton is so hot, I want to kiss him in the...",1300.0,83.0,468.0,617.0
649,Death Stranding 2,NaT,[Kojima Productions],NaN,105.0,105.0,"[Adventure, Shooter]",Not Available,[],3.0,0.0,209.0,644.0
713,Final Fantasy VII Rebirth,2023-12-31,[Square Enix],NaN,192.0,192.0,[],This next standalone chapter in the FINAL FANT...,[],20.0,3.0,354.0,1100.0
719,Lies of P,2023-08-01,"[NEOWIZ, Round8 Studio]",NaN,175.0,175.0,[RPG],"Inspired by the familiar story of Pinocchio, L...",[],5.0,0.0,260.0,939.0
726,Judas,2025-03-31,[Ghost Story Games],NaN,90.0,90.0,"[Adventure, Shooter]",A disintegrating starship. A desperate escape ...,[],1.0,0.0,92.0,437.0
746,Like a Dragon Gaiden: The Man Who Erased His Name,2023-12-31,"[Ryū Ga Gotoku Studios, Sega]",NaN,118.0,118.0,"[Adventure, Brawler, RPG]",This game covers Kiryu's story between Yakuza ...,[],2.0,1.0,145.0,588.0
972,The Legend of Zelda: Tears of the Kingdom,2023-05-12,"[Nintendo, Nintendo EPD Production Group No. 3]",NaN,581.0,581.0,"[Adventure, RPG]",The Legend of Zelda: Tears of the Kingdom is t...,[],72.0,6.0,1600.0,5400.0
1130,Star Wars Jedi: Survivor,2023-04-28,"[Respawn Entertainment, Electronic Arts]",NaN,250.0,250.0,[Adventure],The story of Cal Kestis continues in Star Wars...,[],13.0,2.0,367.0,1400.0
1160,We Love Katamari Reroll + Royal Reverie,2023-06-02,"[Bandai Namco Entertainment, MONKEYCRAFT Co. Ltd]",NaN,51.0,51.0,"[Adventure, Puzzle]",We Love Katamari Reroll + Royal Reverie is a r...,[],3.0,0.0,74.0,291.0


In [17]:
df_games[df_games['Release Date'].isnull()]
# Release Date = NaT (Missing / unknown dates) i.e Games without a finalized release date at the time of data collection
# Rating = NaN (No or/ Not enough user ratings yet). 
# Genres = [] (Empty list [] ≠ NaN; not null) Genres were not assigned at scrape time

# Donot fix as agrregate functions ignore nulls easily and we want to preserve the integrity of the data without making assumptions i.e fake fills


,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist
644,Deltarune,NaT,[tobyfox],4.3,313.0,313.0,"[Adventure, Indie, Music, Puzzle, RPG]","UNDERTALE's parallel story, DELTARUNE. Meet ne...","['Spamton is so hot, I want to kiss him in the...",1300.0,83.0,468.0,617.0
649,Death Stranding 2,NaT,[Kojima Productions],NaN,105.0,105.0,"[Adventure, Shooter]",Not Available,[],3.0,0.0,209.0,644.0
1252,Elden Ring: Shadow of the Erdtree,NaT,"[FromSoftware, Bandai Namco Entertainment]",4.8,18.0,18.0,"[Adventure, RPG]",An expansion to Elden Ring setting players on ...,['I really loved that they integrated Family G...,1.0,0.0,39.0,146.0


In [18]:
df_games['Title'].head(15)

0                                  Elden Ring
1                                       Hades
2     The Legend of Zelda: Breath of the Wild
3                                   Undertale
4                               Hollow Knight
5                                   Minecraft
6                                       Omori
7                               Metroid Dread
8                                    Among Us
9                              NieR: Automata
10                            Persona 5 Royal
11                                      Stray
12                                 God of War
13                                   Portal 2
14                                 Bloodborne
Name: Title, dtype: object

IMPORT GAME SALES DATA TO DATA FRAME

In [19]:
df_gameSales = pd.read_csv('vgsales.csv')
df_gameSales.head() 

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


BASIC DATA EXPLORATION - DATASET 2

In [20]:
df_gameSales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


In [21]:
df_gameSales.value_counts('Genre')

Genre
Action          3316
Sports          2346
Misc            1739
Role-Playing    1488
Shooter         1310
Adventure       1286
Racing          1249
Platform         886
Simulation       867
Fighting         848
Strategy         681
Puzzle           582
Name: count, dtype: int64

In [22]:
df_gameSales['Name'].head(20)

0                                       Wii Sports
1                                Super Mario Bros.
2                                   Mario Kart Wii
3                                Wii Sports Resort
4                         Pokemon Red/Pokemon Blue
5                                           Tetris
6                            New Super Mario Bros.
7                                         Wii Play
8                        New Super Mario Bros. Wii
9                                        Duck Hunt
10                                      Nintendogs
11                                   Mario Kart DS
12                     Pokemon Gold/Pokemon Silver
13                                         Wii Fit
14                                    Wii Fit Plus
15                              Kinect Adventures!
16                              Grand Theft Auto V
17                   Grand Theft Auto: San Andreas
18                               Super Mario World
19    Brain Age: Train Your Bra

# DATA CLEANING 

FIX 1: Handling MISSING VALUE / Nulls

In [23]:
median_year = int(df_gameSales['Year'].median())
df_gameSales['Year'].fillna(median_year, inplace=True)
# Filling missing Year values with the median year of release to maintain data integrity without making assumptions about specific years. 
# Median is used instead of mean to avoid skewing from outliers.

C:\Users\aniru\AppData\Local\Temp\ipykernel_8556\3965755085.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_gameSales['Year'].fillna(median_year, inplace=True)


In [24]:
df_gameSales['Publisher'].fillna('Unknown', inplace=True) 
# Fill missing Publisher values with 'Unknown' to preserve data integrity without making assumptions about specific publishers.

C:\Users\aniru\AppData\Local\Temp\ipykernel_8556\705237474.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_gameSales['Publisher'].fillna('Unknown', inplace=True)


FIX 2:  DATA TYPE CONVERSION

In [25]:
df_gameSales['Year'] = df_gameSales['Year'].astype(int) # Convert Year to integer after filling NaN values 
# reason - Floats break grouping & joins and is slow, while int is fast 

# Validate Sales Metrics (Data Integrity)

In [26]:
# Check for negative or null sales values which are invalid and may indicate data quality issues.
sales_cols = [
    'NA_Sales', 'EU_Sales', 'JP_Sales',
    'Other_Sales', 'Global_Sales'
]

(df_gameSales[sales_cols] < 0).sum()
# No negative sales values found, indicating good data quality in sales columns.

NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64

In [27]:
# Validate Global_Sales consistency
df_gameSales['calc_global'] = (
    df_gameSales['NA_Sales'] +
    df_gameSales['EU_Sales'] +
    df_gameSales['JP_Sales'] +
    df_gameSales['Other_Sales']
)

(df_gameSales['Global_Sales'] - df_gameSales['calc_global']).abs().describe()
# Validate - Global_Sales = the sum of regional sales with minor rounding differences (≤0.02 million units), confirming data integrity.”


count    16598.000000
mean         0.002724
std          0.004466
min          0.000000
25%          0.000000
50%          0.000000
75%          0.010000
max          0.020000
dtype: float64

In [28]:
df_gameSales.drop(columns=['calc_global'], inplace=True)


# FIX: Standardize / Normalize Game Name / Title (JOIN-CRITICAL)

In [29]:
import re 

# This removes: : ' - & ! etc.

def normalize_name(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)   # remove punctuation
    text = re.sub(r'\s+', ' ', text)      # remove extra spaces
    return text.strip()

df_games['Title_norm'] = df_games['Title'].apply(normalize_name)
df_gameSales['Name_norm'] = df_gameSales['Name'].apply(normalize_name)

# Created a cleaned version of the Name / title column for more reliable merging 
# Prevents SQL join mismatches due to inconsistent capitalization or extra spaces in game titles


# Validate Match Quality

In [30]:
common_games = set(df_games['Title_norm']).intersection(
    set(df_gameSales['Name_norm'])
)

len(common_games)


486

In [31]:
print(list(common_games)[:20])

['sin and punishment', 'the witcher 3 wild hunt', 'silent hill', 'digimon world next order', 'halo the master chief collection', 'super mario galaxy 2', 'shin megami tensei nocturne', 'the last of us', 'ms pacman', 'nier', 'yakuza kiwami', 'mirrors edge', 'bully', 'hotel dusk room 215', 'call of duty modern warfare 2', 'xenosaga episode iii also sprach zarathustra', 'metroid prime 2 echoes', 'pacman', 'skate 3', 'earthbound']


In [32]:
df_gameSales[df_gameSales['Name_norm'].isin(common_games)]['Name_norm'].nunique()

# Checks every single row in the Name_clean column of Sales data and returns list of True or/ False values (a Boolean mask) to "filter" the original table with only "True" Records


486

# SAVE THE CLEANED DATASETS FOR PHASE 2

In [33]:
df_games.to_csv('CleanEngagementData.csv', index=False)
df_gameSales.to_csv('CleanSalesData.csv', index=False)

print("Phase_1 Complete: Files saved to Staging area")

Phase_1 Complete: Files saved to Staging area
